# Obtención y consolidación de datos

## Resumen

- **Fuente:** [Buscador de establecimientos educativos del MINEDUC](https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/).
- **Filtro:** `NIVEL ESCOLAR = DIVERSIFICADO`.
- **Cobertura:** 22 departamentos; el buscador consulta `CIUDAD CAPITAL` por separado de `GUATEMALA`.
- **Fecha de extracción:** 2026-07-18.

Los archivos por consulta fueron descargados manualmente debido a las protecciones del sitio. El CSV consolidado comprometido en el repositorio es el ancla inmutable del análisis. Este notebook verifica su estructura, cobertura, filtro y huella SHA-256, y genera un manifiesto reproducible.

## Configuración

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data" / "raw" / "establecimientos_raw.csv").exists():
    raise FileNotFoundError("Ejecute el notebook desde la raíz del repositorio o desde notebooks/.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

from src import catalogos, scraping

## Carga del CSV crudo

In [2]:
RUTA_RAW = ROOT / "data" / "raw" / "establecimientos_raw.csv"
RUTA_MANIFIESTO = ROOT / "data" / "raw" / "manifiesto_fuentes.csv"

df_raw = pd.read_csv(RUTA_RAW, dtype="string", keep_default_na=False)
print(f"Filas: {df_raw.shape[0]:,}")
print(f"Columnas: {df_raw.shape[1]}")
df_raw.head()

Filas: 11,891
Columnas: 17


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,15-01-0050-46,15-01-0890,BAJA VERAPAZ,SALAMA,ESCUELA NORMAL RURAL NO.4 DR. ELIZARDO URIZAR ...,13 AVENIDA 7-41 ZONA 2 BARRIO HACIENDA DE LA V...,79400455,IRIS QUETZALI ORDOÑEZ TURCIOS,ANA ISABEL GUEVARA MEZA,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),BAJA VERAPAZ
1,15-01-0051-46,15-01-0890,BAJA VERAPAZ,SALAMA,COLEGIO PARTICULAR MIXTO TEZULUTLAN,"DIAGONAL 4, 1-24 ZONA 2",79400182,IRIS QUETZALI ORDOÑEZ TURCIOS,ELIA NÍVEA LÓPEZ GARCÍA,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
2,15-01-0052-46,15-01-0895,BAJA VERAPAZ,SALAMA,LICEO MIXTO SAN MATEO,9A. AVENIDA 6-67 ZONA 1,79401926,NELSON JOEL HERNANDEZ HERNANDEZ,CARMEN ALICIA HERNÁNDEZ GUILLERMO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
3,15-01-0087-46,15-01-0890,BAJA VERAPAZ,SALAMA,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,"13 AVENIDA 7-41, ZONA 2, BARRIO HACIENDA DE LA...",53009229,IRIS QUETZALI ORDOÑEZ TURCIOS,ROSALINA NADEYDA TOJ MORENTE,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
4,15-01-0099-46,15-01-0890,BAJA VERAPAZ,SALAMA,COLEGIO PARTICULAR MIXTO TEZULUTLAN,"DIAGONAL 4, 1-24 ZONA 2",79400182,IRIS QUETZALI ORDOÑEZ TURCIOS,ELIA NÍVEA LÓPEZ GARCÍA,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),BAJA VERAPAZ


## Verificación de esquema, nivel y cobertura

In [3]:
COLUMNAS_ESPERADAS = [
    "CODIGO", "DISTRITO", "DEPARTAMENTO", "MUNICIPIO", "ESTABLECIMIENTO",
    "DIRECCION", "TELEFONO", "SUPERVISOR", "DIRECTOR", "NIVEL", "SECTOR",
    "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTAL",
]
assert list(df_raw.columns) == COLUMNAS_ESPERADAS

filas_con_datos = df_raw[df_raw["CODIGO"].str.strip().ne("")]
assert filas_con_datos["NIVEL"].eq("DIVERSIFICADO").all()

departamentos_presentes = set(filas_con_datos["DEPARTAMENTO"].str.strip().str.upper())
faltantes = set(catalogos.DEPARTAMENTOS_OFICIALES) - departamentos_presentes
extras = departamentos_presentes - set(catalogos.DEPARTAMENTOS_OFICIALES)

print("Departamentos oficiales ausentes:", faltantes or "ninguno")
print("Consulta adicional del buscador:", extras)
filas_con_datos["DEPARTAMENTO"].value_counts()

Departamentos oficiales ausentes: ninguno
Consulta adicional del buscador: {'CIUDAD CAPITAL'}


DEPARTAMENTO
CIUDAD CAPITAL    2161
GUATEMALA         1908
SAN MARCOS         724
ESCUINTLA          708
HUEHUETENANGO      591
QUETZALTENANGO     551
PETEN              516
ALTA VERAPAZ       475
SUCHITEPEQUEZ      437
CHIMALTENANGO      435
SACATEPEQUEZ       430
IZABAL             413
JUTIAPA            392
RETALHULEU         364
QUICHE             323
CHIQUIMULA         239
SANTA ROSA         213
SOLOLA             192
JALAPA             183
BAJA VERAPAZ       171
EL PROGRESO        158
ZACAPA             156
TOTONICAPAN        128
Name: count, dtype: Int64

## Verificación reproducible de la unión

In [4]:
import tempfile

filas_vacias = df_raw.apply(lambda serie: serie.str.strip().eq("")).all(axis=1)
posiciones_separadores = df_raw.index[filas_vacias].tolist()
assert len(posiciones_separadores) == 23

bloques = []
inicio = 0
for fin in posiciones_separadores:
    bloques.append(df_raw.loc[inicio:fin].reset_index(drop=True))
    inicio = fin + 1

with tempfile.TemporaryDirectory() as directorio_temporal:
    rutas_bloques = []
    for numero, bloque in enumerate(bloques, start=1):
        ruta = Path(directorio_temporal) / f"consulta_{numero:02d}.csv"
        bloque.to_csv(ruta, index=False, lineterminator="\n")
        rutas_bloques.append(ruta)

    reconstruido = scraping.unir_csvs(rutas_bloques)

pd.testing.assert_frame_equal(df_raw, reconstruido, check_dtype=False)
print("Bloques unidos:", len(bloques))
print("La unión reproduce exactamente las 11,891 filas del CSV crudo.")

Bloques unidos: 23
La unión reproduce exactamente las 11,891 filas del CSV crudo.


## Manifiesto y huella de integridad

In [5]:
manifiesto = scraping.crear_manifiesto_consolidado(RUTA_RAW, "2026-07-18")
manifiesto.to_csv(RUTA_MANIFIESTO, index=False, lineterminator="\n")

print("SHA-256:", scraping.sha256_archivo(RUTA_RAW))
print("Registros documentados:", int(manifiesto["registros"].sum()))
manifiesto

SHA-256: bbc1fa3b26b2509a22d547dc81dfa3b8bbc470f33c26b210de5b448d8d1a1d1c
Registros documentados: 11868


,consulta_departamento,nivel,registros,fecha_extraccion,archivo_consolidado,sha256_consolidado
0,ALTA VERAPAZ,DIVERSIFICADO,475,2026-07-18,establecimientos_raw.csv,bbc1fa3b26b2509a22d547dc81dfa3b8bbc470f33c26b2...
1,BAJA VERAPAZ,DIVERSIFICADO,171,2026-07-18,establecimientos_raw.csv,bbc1fa3b26b2509a22d547dc81dfa3b8bbc470f33c26b2...
2,CHIMALTENANGO,DIVERSIFICADO,435,2026-07-18,establecimientos_raw.csv,bbc1fa3b26b2509a22d547dc81dfa3b8bbc470f33c26b2...
3,CHIQUIMULA,DIVERSIFICADO,239,2026-07-18,establecimientos_raw.csv,bbc1fa3b26b2509a22d547dc81dfa3b8bbc470f33c26b2...
4,CIUDAD CAPITAL,DIVERSIFICADO,2161,2026-07-18,establecimientos_raw.csv,bbc1fa3b26b2509a22d547dc81dfa3b8bbc470f33c26b2...
5,EL PROGRESO,DIVERSIFICADO,158,2026-07-18,establecimientos_raw.csv,bbc1fa3b26b2509a22d547dc81dfa3b8bbc470f33c26b2...
6,ESCUINTLA,DIVERSIFICADO,708,2026-07-18,establecimientos_raw.csv,bbc1fa3b26b2509a22d547dc81dfa3b8bbc470f33c26b2...
7,GUATEMALA,DIVERSIFICADO,1908,2026-07-18,establecimientos_raw.csv,bbc1fa3b26b2509a22d547dc81dfa3b8bbc470f33c26b2...
8,HUEHUETENANGO,DIVERSIFICADO,591,2026-07-18,establecimientos_raw.csv,bbc1fa3b26b2509a22d547dc81dfa3b8bbc470f33c26b2...
9,IZABAL,DIVERSIFICADO,413,2026-07-18,establecimientos_raw.csv,bbc1fa3b26b2509a22d547dc81dfa3b8bbc470f33c26b2...


## Nota de reproducibilidad

`src/scraping.py` contiene funciones para descargar una consulta y unir una lista de CSV con el mismo esquema. La descarga histórica exacta no puede repetirse desde una fuente viva; por eso la fecha y el hash fijan el CSV crudo utilizado. A partir de este archivo, todo el diagnóstico y la limpieza sí deben reproducirse de extremo a extremo.